In [1]:
import requests
import pandas as pd
from datetime import datetime
import json
import time

In [2]:
def search_grants(keyword="", 
                  agencies="", 
                  opp_statuses="forecasted|posted", 
                  funding_categories="",
                  eligibilities="",
                  aln="",
                  rows=100,
                  fetch_details=False):
    """
    Search for grants on grants.gov
    
    Parameters:
    - keyword: Search keyword (e.g., "technology", "education")
    - agencies: Agency codes separated by | (e.g., "HHS|DOE")
    - opp_statuses: Status of opportunities separated by | 
                    Options: "forecasted", "posted", "closed", "archived"
                    Default: "forecasted|posted"
    - funding_categories: Category codes (e.g., "HL" for Health, "ED" for Education)
    - eligibilities: Eligibility codes
    - aln: Assistance Listing Number
    - rows: Number of results to return (default 100)
    - fetch_details: If True, fetch detailed info for each grant (includes award amounts)
                     WARNING: This makes 1 API call per grant and will be MUCH slower
    
    Returns:
    - DataFrame with grant opportunities
    """
    
    url = "https://api.grants.gov/v1/api/search2"
    
    # Prepare request body
    payload = {
        "rows": rows,
        "keyword": keyword,
        "oppNum": "",
        "eligibilities": eligibilities,
        "agencies": agencies,
        "oppStatuses": opp_statuses,
        "aln": aln,
        "fundingCategories": funding_categories
    }
    
    headers = {
        "Content-Type": "application/json"
    }
    
    try:
        print(f"Searching grants.gov with parameters:")
        print(f"  Keyword: {keyword if keyword else 'None'}")
        print(f"  Agencies: {agencies if agencies else 'All'}")
        print(f"  Status: {opp_statuses}")
        print(f"  Max results: {rows}")
        print(f"  Fetch details: {fetch_details}\n")
        
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        
        data = response.json()
        
        # Check for errors
        if data.get('errorcode', 1) != 0:
            print(f"API Error: {data.get('msg', 'Unknown error')}")
            return pd.DataFrame()
        
        # Extract opportunities
        opportunities = data.get('data', {}).get('oppHits', [])
        hit_count = data.get('data', {}).get('hitCount', 0)
        
        print(f"Found {hit_count} total grants")
        print(f"Retrieved {len(opportunities)} grants\n")
        
        if not opportunities:
            print("No grants found matching criteria")
            return pd.DataFrame()
        
        # Convert to DataFrame
        df = pd.DataFrame(opportunities)
        
        # Convert ALN list to string if present
        if 'alnist' in df.columns:
            df['aln_numbers'] = df['alnist'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')
            df = df.drop('alnist', axis=1)
        
        print(f"Summary columns: {', '.join(df.columns.tolist())}\n")
        
        # Fetch detailed information if requested
        if fetch_details:
            print("Fetching detailed information for each grant...")
            print("This may take a while...\n")
            
            detailed_data = []
            for idx, row in df.iterrows():
                opp_id = row['id']
                print(f"Fetching details for grant {idx+1}/{len(df)} (ID: {opp_id})...", end='\r')
                
                details = fetch_opportunity_details(opp_id)
                if details:
                    detailed_data.append(details)
                
                # Be nice to the API - small delay between requests
                time.sleep(0.5)
            
            print("\n")
            
            if detailed_data:
                details_df = pd.DataFrame(detailed_data)
                # Merge with summary data
                df = df.merge(details_df, left_on='id', right_on='opportunityId', how='left')
                print(f"All columns (with details): {', '.join(df.columns.tolist())}\n")
        
        return df
        
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return pd.DataFrame()
    except json.JSONDecodeError as e:
        print(f"JSON decode error: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Unexpected error: {e}")
        return pd.DataFrame()


def fetch_opportunity_details(opportunity_id):
    """
    Fetch detailed information for a specific grant opportunity
    
    Parameters:
    - opportunity_id: The ID of the opportunity
    
    Returns:
    - Dictionary with detailed information including award amounts
    """
    url = "https://api.grants.gov/v1/api/fetchOpportunity"
    
    payload = {
        "opportunityId": opportunity_id
    }
    
    headers = {
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        
        data = response.json()
        
        if data.get('errorcode', 1) != 0:
            return None
        
        opp_data = data.get('data', {})
        synopsis = opp_data.get('synopsis', {})
        
        # Extract key fields
        details = {
            'opportunityId': opportunity_id,
            'awardCeiling': synopsis.get('awardCeiling', ''),
            'awardFloor': synopsis.get('awardFloor', ''),
            'costSharing': synopsis.get('costSharing', ''),
            'synopsisDesc': synopsis.get('synopsisDesc', ''),
            'agencyContactName': synopsis.get('agencyContactName', ''),
            'agencyContactEmail': synopsis.get('agencyContactEmail', ''),
            'agencyContactPhone': synopsis.get('agencyContactPhone', ''),
            'estimatedTotalFunding': synopsis.get('estimatedTotalFunding', ''),
            'expectedNumberOfAwards': synopsis.get('expectedNumberOfAwards', ''),
        }
        
        return details
        
    except Exception as e:
        return None


def export_to_excel(df, filename=None):
    """
    Export DataFrame to Excel file
    
    Parameters:
    - df: DataFrame to export
    - filename: Output filename (default: grants_YYYYMMDD_HHMMSS.xlsx)
    """
    
    if df.empty:
        print("No data to export")
        return
    
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"grants_{timestamp}.xlsx"
    
    try:
        # Export to Excel
        df.to_excel(filename, index=False, engine='openpyxl')
        print(f"✓ Successfully exported {len(df)} grants to {filename}")
        print(f"  Columns: {', '.join(df.columns.tolist())}")
        
    except Exception as e:
        print(f"Error exporting to Excel: {e}")


def display_sample(df, num_rows=3):
    """
    Display sample rows from DataFrame, showing whatever columns are available
    """
    if df.empty:
        print("No data to display")
        return
    
    print(f"\nSample results (showing {min(num_rows, len(df))} of {len(df)} grants):")
    print("-" * 80)
    
    # Display all columns for the sample rows
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 50)
    
    print(df.head(num_rows).to_string(index=False))
    print()


In [6]:

# Example 1: Quick search WITHOUT detailed info (fast)
print("=" * 80)
print("EXAMPLE 1: Quick search for health grants (summary only)")
print("=" * 80)
df = search_grants(
    keyword="health",
    opp_statuses="posted",
    rows=10,
    fetch_details=False  # Fast - no award amounts
)

if not df.empty:
    display_sample(df, num_rows=3)
    export_to_excel(df, "health_grants_summary.xlsx")

print("\n")

# Example 2: Search WITH detailed info including award amounts (slow)
print("=" * 80)
print("EXAMPLE 2: Technology grants WITH award amounts (detailed)")
print("=" * 80)
df = search_grants(
    keyword="technology",
    opp_statuses="posted",
    rows=5,  # Fewer grants because this is slow
    fetch_details=True  # Slow - includes award ceiling/floor
)

if not df.empty:
    # Show columns with award info
    award_cols = ['number', 'title', 'awardFloor', 'awardCeiling', 'costSharing']
    available_cols = [col for col in award_cols if col in df.columns]
    if available_cols:
        print("\nAward information:")
        print(df[available_cols].head())
    
    export_to_excel(df, "technology_grants_detailed.xlsx")

print("\n")

# Example 3: Custom search
print("=" * 80)
print("EXAMPLE 3: Custom search")
print("=" * 80)
df = search_grants(
    keyword="small business",
    agencies="SBA",
    opp_statuses="posted",
    rows=10,
    fetch_details=True  # Get award amounts
)

if not df.empty:
    export_to_excel(df, "sba_grants_detailed.xlsx")

EXAMPLE 1: Quick search for health grants (summary only)
Searching grants.gov with parameters:
  Keyword: health
  Agencies: All
  Status: posted
  Max results: 10
  Fetch details: False



Found 302 total grants
Retrieved 10 grants

Summary columns: id, number, title, agencyCode, agency, openDate, closeDate, oppStatus, docType, cfdaList


Sample results (showing 3 of 10 grants):
--------------------------------------------------------------------------------
    id            number                                                                                                                    title    agencyCode                              agency   openDate  closeDate oppStatus  docType         cfdaList
356250        PAR-24-281 NLM Information Resource Grants to Reduce Health Disparities and Promote Health for All (G08 Clinical Trial Not Allowed)     HHS-NIH11       National Institutes of Health 08/28/2024 05/25/2026    posted synopsis [93.879, 93.313]
341131     NAP-AX-22-001                                               Leading Edge Acceleration Projects (LEAP) in Health Information Technology    HHS-OS-ONC  Office of the National Coordinator 06/14/2022 09/30/2027 